In [2]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import psycopg2
from minio import Minio

spark = SparkSession.builder \
        .appName("Brazilian-Ecommerce") \
        .master("spark://spark-master:7077") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.hadoop:hadoop-aws:3.3.4") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minio_admin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minio_secure_pwd") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.executor.memory", "1g") \
        .config("spark.driver.memory", "1g") \
        .getOrCreate()

# set log level
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark Session created successfully")
print(f"📊 Spark Version: {spark.version}")
print(f"🔧 Application Name: {spark.sparkContext.appName}")
print(f"🎯 Master URL: {spark.sparkContext.master}")

✅ Spark Session created successfully
📊 Spark Version: 3.5.0
🔧 Application Name: Brazilian-Ecommerce
🎯 Master URL: spark://spark-master:7077


In [ ]:
#1 read file from minIO

In [6]:
bronze_paths = {
    "customers": "s3a://raw-data/olist_customers_dataset.csv",
    "geolocation": "s3a://raw-data/olist_geolocation_dataset.csv",
    "order_items": "s3a://raw-data/olist_order_items_dataset.csv",
    "payments": "s3a://raw-data/olist_order_payments_dataset.csv",
    "reviews": "s3a://raw-data/olist_order_reviews_dataset.csv",
    "orders": "s3a://raw-data/olist_orders_dataset.csv",
    "products": "s3a://raw-data/olist_products_dataset.csv",
    "sellers": "s3a://raw-data/olist_sellers_dataset.csv",
    "category_name_translation" : "s3a://raw-data/product_category_name_translation.csv",
}

#todo : read each tables into dict dataframe
dfs = {}

for name,path in bronze_paths.items():
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)

for name, df in dfs.items():
    count = df.count()
    
    print(f"📄 CSV {name} read Performance:")
    print(f"   Records: {count:,}")
    print(f"\nSchema:")
    df.printSchema()
    print("=========================")
            
            


📄 CSV customers read Performance:
   Records: 99,441

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: integer (nullable = true)

📄 CSV geolocation read Performance:
   Records: 1,000,163

Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

📄 CSV order_items read Performance:
   Records: 112,650

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_da

In [ ]:
#3 . schema definition customers


In [49]:
schemas = {
    "customers": StructType([
    StructField("customer_id",StringType(),False),
    StructField("customer_unique_id",StringType(),False),
    StructField("customer_zip_code_prefix",IntegerType(),False),
    StructField("customer_city",StringType(),False),
    StructField("customer_state",StringType(),False),
    
]),
    "geolocation": StructType([
        StructField("geolocation_zip_code_prefix", IntegerType(), True),
        StructField("geolocation_lat", DoubleType(), True),
        StructField("geolocation_lng", DoubleType(), True),
        StructField("geolocation_city", StringType(), True),
        StructField("geolocation_state", StringType(), True),
    ]),

    "order_items": StructType([
        StructField("order_id", StringType(), True),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), True),
        StructField("seller_id", StringType(), True),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True),
    ]),

    "payments": StructType([
        StructField("order_id", StringType(), True),
        StructField("payment_sequential", IntegerType(), True),
        StructField("payment_type", StringType(), True),
        StructField("payment_installments", IntegerType(), True),
        StructField("payment_value", DoubleType(), True),
    ]),

    "orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True),
    ]),

    "products": StructType([
        StructField("product_id", StringType(), True),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", IntegerType(), True),
        StructField("product_length_cm", IntegerType(), True),
        StructField("product_height_cm", IntegerType(), True),
        StructField("product_width_cm", IntegerType(), True),
    ]),

    "sellers": StructType([
        StructField("seller_id", StringType(), True),
        StructField("seller_zip_code_prefix", IntegerType(), True),
        StructField("seller_city", StringType(), True),
        StructField("seller_state", StringType(), True),
    ]),

    "category_name_translation": StructType([
        StructField("product_category_name", StringType(), True),
        StructField("product_category_name_english", StringType(), True),
    ]),
}



dfs_schema = {}

for name,schema in schemas.items():
    dfs_schema[name] = spark.read \
                        .schema(schema) \
                        .csv(bronze_paths[name],header=True)


# kiểm tra nhanh toàn bộ
for name, df in dfs_schema.items():
    print(f"--- {name} ---")
    df.printSchema()
    print(f"Rows: {df.count()}\n")



--- reviews ---
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

Rows: 99227

--- customers ---
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Rows: 99441

--- geolocation ---
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

Rows: 1000163

--- order_items ---
root
 |-- order_id: string (nu

In [ ]:
#4 process reviews file

In [ ]:
# add multiLine for reviews 
reviews_schema=  StructType([
        StructField("review_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("review_score", IntegerType(), True),
        StructField("review_comment_title", StringType(), True),
        StructField("review_comment_message", StringType(), True),
        StructField("review_creation_date", TimestampType(), True),
        StructField("review_answer_timestamp", TimestampType(), True),
    ])
df_reviews = spark.read \
    .schema(reviews_schema) \
    .option("multiLine",True) \
    .csv(bronze_paths["reviews"],header=True) 
   

print(f"--- reviews ---")
df_reviews.printSchema()
print(f"Rows: {df_reviews.count()}\n")